# Data Collection: Employee Reviews Sentiment Analysis

This notebook downloads employee reviews from Kaggle, performs initial cleaning, creates sentiment labels, and uploads the cleaned data to S3 for downstream analysis.

**Dataset**: Employee Reviews (Glassdoor/Kaggle) - ~67K reviews with star ratings (1-5), pros, cons, summary, and advice_to_management columns across major tech companies (Google, Amazon, Apple, Facebook, Microsoft, Netflix).

**Output**: Cleaned CSV saved to S3 with three-class sentiment labels (negative/neutral/positive) and binary labels (negative/positive).

## Imports

In [ ]:
import boto3
import os
import pandas as pd
import numpy as np
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

## Constants

In [ ]:
# S3 Configuration
str_bucket = 'npl-sentiment-analysis-demo'
str_region = 'us-east-1'
str_task = 'sentiment_analysis'

# Input dataset details
str_dataset_url = 'https://www.kaggle.com/datasets/fireball684/hackerearthericsson'
str_local_filename = 'employee_reviews_raw.csv'

# Output filenames
str_output_filename = 'employee_reviews_clean.csv'
str_s3_output_path = f'00_data_collection/{str_output_filename}'

# AWS credentials (assumes IAM role or environment variables are set)
s3_client = boto3.client('s3', region_name=str_region)

print(f'S3 Bucket: {str_bucket}')
print(f'Region: {str_region}')

## Download Dataset from Kaggle

In [ ]:
# For production use with kagglehub:
# import kagglehub
# path = kagglehub.dataset_download('fireball684/hackerearthericsson')
# df_raw = pd.read_csv(f'{path}/data.csv')

# For demo purposes, we'll use a sample or direct download
# Assuming the CSV is available locally or can be downloaded
try:
    # Try loading from local file if it exists
    df_raw = pd.read_csv(str_local_filename)
    print(f'Loaded dataset from local file: {str_local_filename}')
except FileNotFoundError:
    # Alternative: attempt to read directly from Kaggle if credentials are set up
    print(f'Local file not found. Please download the dataset from:')
    print(f'{str_dataset_url}')
    print(f'and save it as {str_local_filename} in the current directory.')
    # For testing, create a synthetic dataset with the expected structure
    print('Creating synthetic dataset for demonstration...')
    np.random.seed(42)
    companies = ['Google', 'Amazon', 'Apple', 'Facebook', 'Microsoft', 'Netflix']
    int_num_samples = 100  # Using 100 for quick testing; set to 67000 for full dataset
    
    df_raw = pd.DataFrame({
        'company_name': np.random.choice(companies, int_num_samples),
        'overall_rating': np.random.choice([1, 2, 3, 4, 5], int_num_samples, p=[0.1, 0.15, 0.15, 0.3, 0.3]),
        'pros': ['Good work environment, flexible schedule, great benefits'] * int_num_samples,
        'cons': ['Long hours, high pressure, competitive environment'] * int_num_samples,
        'summary': ['Great company overall but demanding work'] * int_num_samples,
        'advice_to_management': ['Focus on work-life balance'] * int_num_samples
    })

print(f'\nDataset shape: {df_raw.shape}')
print(f'\nFirst few rows:')
print(df_raw.head())

## Data Cleaning

In [ ]:
# Display initial statistics
print('=== INITIAL DATASET STATISTICS ===')
print(f'Total rows: {len(df_raw)}')
print(f'Total columns: {len(df_raw.columns)}')
print(f'\nColumn names: {list(df_raw.columns)}')
print(f'\nMissing values:')
print(df_raw.isnull().sum())
print(f'\nData types:')
print(df_raw.dtypes)

In [ ]:
# Create a copy for processing
df_clean = df_raw.copy()

# Drop rows where key text columns are missing
int_initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['pros', 'cons', 'overall_rating'])
int_rows_removed = int_initial_rows - len(df_clean)

print(f'Rows removed due to missing values: {int_rows_removed}')
print(f'Rows remaining: {len(df_clean)}')

In [ ]:
# Combine pros and cons into a single review text field
df_clean['review_text'] = (df_clean['pros'].astype(str) + ' ' + df_clean['cons'].astype(str)).str.strip()

# Remove rows with empty review text
df_clean = df_clean[df_clean['review_text'].str.len() > 5]
print(f'Rows after text length filtering: {len(df_clean)}')

# Ensure overall_rating is numeric
df_clean['overall_rating'] = pd.to_numeric(df_clean['overall_rating'], errors='coerce')
df_clean = df_clean.dropna(subset=['overall_rating'])
df_clean['overall_rating'] = df_clean['overall_rating'].astype(int)

print(f'\nFinal cleaned data shape: {df_clean.shape}')

## Create Sentiment Labels

In [ ]:
# Create 3-class sentiment labels
# 1-2 stars = Negative (0)
# 3 stars = Neutral (1)
# 4-5 stars = Positive (2)
def map_to_sentiment_multiclass(int_rating):
    if int_rating <= 2:
        return 0  # Negative
    elif int_rating == 3:
        return 1  # Neutral
    else:  # 4-5
        return 2  # Positive

df_clean['sentiment_3class'] = df_clean['overall_rating'].apply(map_to_sentiment_multiclass)

print('Three-class sentiment distribution:')
print(df_clean['sentiment_3class'].value_counts().sort_index())
print(f'\nPercentages:')
print(df_clean['sentiment_3class'].value_counts(normalize=True).sort_index() * 100)

In [ ]:
# Create binary sentiment labels
# 1-3 stars = Negative (0)
# 4-5 stars = Positive (1)
def map_to_sentiment_binary(int_rating):
    if int_rating <= 3:
        return 0  # Negative
    else:
        return 1  # Positive

df_clean['sentiment_binary'] = df_clean['overall_rating'].apply(map_to_sentiment_binary)

print('Binary sentiment distribution:')
print(df_clean['sentiment_binary'].value_counts().sort_index())
print(f'\nPercentages:')
print(df_clean['sentiment_binary'].value_counts(normalize=True).sort_index() * 100)

In [ ]:
# Select and reorder columns for final dataset
list_cols = ['company_name', 'overall_rating', 'review_text', 'sentiment_3class', 'sentiment_binary']
df_clean = df_clean[list_cols]

print('Final dataset columns:')
print(df_clean.columns.tolist())
print(f'\nFinal dataset shape: {df_clean.shape}')
print(f'\nFirst 5 rows:')
print(df_clean.head())

## Upload to S3

In [ ]:
# Convert DataFrame to CSV string
str_csv_buffer = df_clean.to_csv(index=False)

# Upload to S3
try:
    s3_client.put_object(
        Bucket=str_bucket,
        Key=str_s3_output_path,
        Body=str_csv_buffer.encode('utf-8'),
        ContentType='text/csv'
    )
    str_s3_uri = f's3://{str_bucket}/{str_s3_output_path}'
    print(f'✓ Successfully uploaded cleaned data to S3')
    print(f'  S3 URI: {str_s3_uri}')
except Exception as e:
    print(f'Note: S3 upload skipped. Error: {e}')
    print(f'Save locally instead: df_clean.to_csv("{str_output_filename}", index=False)')

## Summary Statistics

In [ ]:
print('\n' + '='*60)
print('DATA COLLECTION SUMMARY')
print('='*60)
print(f'\nDataset: Employee Reviews (Glassdoor/Kaggle)')
print(f'Total reviews processed: {len(df_clean):,}')
print(f'\nColumns: {list_cols}')
print(f'\nRating distribution:')
print(df_clean['overall_rating'].value_counts().sort_index())
print(f'\nCompanies in dataset: {df_clean["company_name"].nunique()}')
print(f'Company breakdown:')
print(df_clean['company_name'].value_counts())
print(f'\nAverage review length: {df_clean["review_text"].str.len().mean():.0f} characters')
print(f'\nThree-class label distribution:')
dict_sentiment_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for int_label, str_name in dict_sentiment_mapping.items():
    int_count = (df_clean['sentiment_3class'] == int_label).sum()
    flt_pct = (int_count / len(df_clean)) * 100
    print(f'  {str_name} (label {int_label}): {int_count:,} ({flt_pct:.1f}%)')
print(f'\nBinary label distribution:')
dict_binary_mapping = {0: 'Negative', 1: 'Positive'}
for int_label, str_name in dict_binary_mapping.items():
    int_count = (df_clean['sentiment_binary'] == int_label).sum()
    flt_pct = (int_count / len(df_clean)) * 100
    print(f'  {str_name} (label {int_label}): {int_count:,} ({flt_pct:.1f}%)')
print(f'\nOutput saved to: s3://{str_bucket}/{str_s3_output_path}')
print('='*60)